# SC4020 Project 2 Task 2: Recommending Targets for a Ligand (Collaborative Filtering)

**Users = ligands, items = targets.** The question is *"for this ligand, which targets will it bind most strongly?"*, so each ligand plays the role of a user and its binding scores play the role of ratings on targets (items).

This notebook needs `data/clean/` from `task1_analysis.ipynb`. Scores: DAVIS $\mathrm{p}K_d$ and KIBA supplied scores (higher = stronger for both).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

SEED = 42                      
CLEAN_DIR = Path("data/clean")
MODEL_DIR = Path("saved_models")
FIG_DIR = Path("figures")
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

SCALES = {
    "davis": {"score_name": "pKd", "units": "pKd = 9 - log10(Kd [nM])", "screening_limit": 5.0},
    "kiba":  {"score_name": "KIBA score", "units": "supplied KIBA scale (higher = stronger)", "screening_limit": np.nan},
}
pairs = {name: pd.read_csv(CLEAN_DIR / f"{name}_pairs.csv", dtype={"ligand_key": str, "target_key": str})
         for name in SCALES}
{name: len(df) for name, df in pairs.items()}

{'davis': 25772, 'kiba': 117657}

## Step 1: Train / validation / test split

Pairs are shuffled with a fixed seed and split 70 / 15 / 15. Each pair lands in exactly one partition. CF cannot predict for a ligand or target it has never seen, so any validation/test pair whose ligand or target is missing from training is moved into training (this is repeated until no such pair remains).

In [2]:
from project2_cf import split_pairs

splits, split_rows = {}, []
for name, df in pairs.items():
    splits[name], moved = split_pairs(df)
    counts = splits[name]["split"].value_counts()
    split_rows.append({"dataset": name.upper(), **{s: counts[s] for s in ["train", "val", "test"]},
                       "moved to train (unseen ligand/target)": moved})

# checks required
for name, df in splits.items():
    assert not df.duplicated(["ligand_key", "target_key"]).any()          # each pair in one partition only
    tr = df[df["split"] == "train"]
    ev = df[df["split"] != "train"]
    assert ev["ligand_key"].isin(tr["ligand_key"]).all() and ev["target_key"].isin(tr["target_key"]).all()

split_table = pd.DataFrame(split_rows).set_index("dataset")
split_table["train %"] = (100 * split_table["train"] / split_table[["train", "val", "test"]].sum(axis=1)).round(1)
split_table

,train,val,test,moved to train (unseen ligand/target),train %
dataset,,,,,
DAVIS,18040,3865,3867,0,70.0
KIBA,82359,17648,17650,0,70.0


## Step 2: The models (adapted from lecture 11.3)

**Baseline predictor (global effects):**

$$b_{ui} = \mu + b_u + b_i, \qquad b_u = \bar r_u - \mu, \qquad b_i = \bar r_i - \mu$$

$\mu$ = mean training score, $\bar r_u$ = mean training score of ligand $u$, $\bar r_i$ = mean training score of target $i$.

**Similarity: centred cosine (according to slide 9).** Subtract each target's mean from its scores, treat unmeasured entries as 0 and then take the cosine between two targets' score vectors:

$$s_{ij} = \frac{\sum_u (r_{ui} - \bar r_i)(r_{uj} - \bar r_j)}{\sqrt{\sum_u (r_{ui} - \bar r_i)^2}\,\sqrt{\sum_u (r_{uj} - \bar r_j)^2}}$$

**Prediction: baseline + neighbour deviations.** Item-based (target–target) version:

$$\hat r_{ui} = b_{ui} + \frac{\sum_{j \in N_k(i;u)} s_{ij}\,(r_{uj} - b_{uj})}{\sum_{j \in N_k(i;u)} s_{ij}}$$

$N_k(i;u)$ = the $k$ targets most similar to $i$ that ligand $u$ has a training score for. User-based (ligand–ligand) CF is the same with the roles of ligands and targets swapped.

Only neighbours with positive similarity are used. With negative weights, the denominator $\sum s_{ij}$ can approach zero and blow up the prediction.

**Fallback:** if a pair has no positive-similarity neighbour, the prediction is the baseline $b_{ui}$.

Numerical safeguard: centred profiles whose norm is within floating-point roundoff of zero have zero similarity. Otherwise constant KIBA profiles can acquire spurious large cosine similarities after normalization. The tolerance scales with score magnitude and the number of observations.


In [3]:
from project2_cf import (build_index, to_idx, fit_baseline, centred_cosine, fit_cf, predict)


## Step 3: Metrics and fair treatment of ties

MSE measures score error. **Pooled CI** compares all observed pairs, including pairs from different ligands; it can be high for a ligand-mean model that cannot rank targets for any one ligand. **Macro CI** averages CI separately over rankable ligands and directly tests within-ligand ordering. Tied predictions receive half credit in CI.

**Hit@1/5 and the expected rank of a true-best target** use exact expectations under uniform random ordering within predicted-score ties. Any target tied for the highest true score counts as correct. This makes the results independent of row order. If a tie block has m candidates, b truly best candidates, and a candidates preceding it, the expected first correct rank is a + (m+1)/(b+1). Hit probabilities use sampling without replacement within that block.

Ligands with fewer than two distinct true scores cannot meaningfully be ranked; they are excluded from ranking/macro-CI averages and their number is reported. MSE and pooled CI retain all observations. Random-ranking expectations account for candidate-set size and multiple true-best targets.

The implementation and small exhaustive checks are in `project2_metrics.py` and `tests/test_metrics.py`.


In [4]:
from project2_metrics import concordance_index, ranking_metrics, summarize_predictions
from project2_cf import evaluate


## Step 4: Hyperparameter selection (validation set only)

Grid search over CF direction (item-based / user-based) and neighbourhood size $k$, subsequently choosing the lowest validation MSE. The test set is not touched here.

In [5]:
KS = [2, 3, 5, 10, 20, 40, 80]

best_params, search_logs = {}, {}
for name, df in splits.items():
    train, val = df[df["split"] == "train"], df[df["split"] == "val"]
    ligands, targets = build_index(df)
    rows = []
    for method in ["item", "user"]:
        model = fit_cf(train, ligands, targets, method)      # similarity does not depend on k
        for k in KS:
            model["k"] = k
            pred, _ = predict(model, val["ligand_key"], val["target_key"])
            rows.append({"method": method, "k": k, "val MSE": np.mean((val["score"].values - pred) ** 2)})
    log = pd.DataFrame(rows).sort_values("val MSE")
    search_logs[name] = log
    best = log.iloc[0]
    best_params[name] = {"method": best["method"], "k": int(best["k"])}
    print(f"{name.upper()}: best CF = {best_params[name]}, val MSE = {best['val MSE']:.4f}")

pd.concat({n.upper(): log.head(5).reset_index(drop=True) for n, log in search_logs.items()})

DAVIS: best CF = {'method': 'item', 'k': 10}, val MSE = 0.3089


KIBA: best CF = {'method': 'user', 'k': 10}, val MSE = 0.2237


method   k   val MSE
DAVIS 0   item  10  0.308855
      1   item   5  0.314608
      2   item  20  0.321692
      3   user  10  0.325322
      4   user   5  0.328046
KIBA  0   user  10  0.223652
      1   user   5  0.225688
      2   user  20  0.229226
      3   user   3  0.235797
      4   user  40  0.239019

In [6]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for ax, (name, log) in zip(axes, search_logs.items()):
    for method, g in log.groupby("method"):
        g = g.sort_values("k")
        ax.plot(g["k"], g["val MSE"], marker="o", label=f"{method}-based")
    ax.set_xscale("log")
    ax.set_xlabel("Neighbourhood size k")
    ax.set_ylabel("Validation MSE")
    ax.set_title(name.upper())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "cf_hyperparameter_search.png", dpi=200)
plt.show()

C:\Users\ROG\AppData\Local\Temp\ipykernel_8016\3852553196.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: Final evaluation

Each model is fitted on training data only with the selected hyperparameters, then evaluated on train, validation and **test (once)**.

The baselines answer *"is CF worth it?"*:
- **global mean**: same score for everything
- **ligand mean**: the fallback which gives every target of a ligand the same score, so it cannot rank targets at all
- **baseline** $\mu + b_u + b_i$: ranks targets by how strongly they bind on average (a "popularity" ranking).

A pure lookup of known measurements has **no answer** for any validation/test pair since none of them appear in training.

In [7]:
final_models, results, scored_test = {}, [], {}
for name, df in splits.items():
    train = df[df["split"] == "train"]
    ligands, targets = build_index(df)
    p = best_params[name]
    final_name = f"CF ({p['method']}-based, k={p['k']})"
    models = {
        "global mean": fit_cf(train, ligands, targets, "global"),
        "ligand mean": fit_cf(train, ligands, targets, "ligand_mean"),
        "baseline (mu + b_u + b_i)": fit_cf(train, ligands, targets, "baseline"),
        final_name: fit_cf(train, ligands, targets, **p),
    }
    final_models[name] = models[final_name]
    for model_name, model in models.items():
        for part in ["train", "val", "test"]:
            metrics, scored = evaluate(model, df[df["split"] == part])
            results.append({"dataset": name.upper(), "model": model_name, "split": part, **metrics})
            if part == "test" and model is final_models[name]:
                scored_test[name] = scored

results_df = pd.DataFrame(results)
results_df[results_df["split"] == "test"].set_index(["dataset", "model"]).drop(columns="split").round(4)

MSE      CI  macro CI   Hit@1   Hit@5  \
dataset model                                                                 
DAVIS   global mean                0.5942  0.5000    0.5000  0.0177  0.0884   
        ligand mean                0.4416  0.7517    0.5000  0.0177  0.0884   
        baseline (mu + b_u + b_i)  0.4071  0.7885    0.6746  0.1212  0.3182   
        CF (item-based, k=10)      0.2774  0.8443    0.7819  0.2273  0.5303   
KIBA    global mean                0.6887  0.5000    0.5000  0.1911  0.6199   
        ligand mean                0.4314  0.7020    0.5000  0.1911  0.6199   
        baseline (mu + b_u + b_i)  0.3760  0.7781    0.6638  0.3395  0.7898   
        CF (user-based, k=10)      0.2202  0.8456    0.8018  0.5673  0.9068   

                                   random Hit@1  random Hit@5  \
dataset model                                                   
DAVIS   global mean                      0.0177        0.0884   
        ligand mean                      0.0177        0.0884   
        baseline (mu + b_u + b_i)        0.0177        0.0884   
        CF (item-based, k=10)            0.0177        0.0884   
KIBA    global mean                      0.1911        0.6199   
        ligand mean                      0.1911        0.6199   
        baseline (mu + b_u + b_i)        0.1911        0.6199   
        CF (user-based, k=10)            0.1911        0.6199   

                                   mean rank of best  ranked ligands  \
dataset model                                                          
DAVIS   global mean                          29.0909              66   
        ligand mean                          29.0909              66   
        baseline (mu + b_u + b_i)            17.1364              66   
        CF (item-based, k=10)                 7.2879              66   
KIBA    global mean                           5.8097            1470   
        ligand mean                           5.8097            1470   
        baseline (mu + b_u + b_i)             3.8728            1470   
        CF (user-based, k=10)                 2.3571            1470   

                                   excluded ligands  fallback %  
dataset model                                                    
DAVIS   global mean                               2         NaN  
        ligand mean                               2         NaN  
        baseline (mu + b_u + b_i)                 2         NaN  
        CF (item-based, k=10)                     2      0.0000  
KIBA    global mean                             504         NaN  
        ligand mean                             504         NaN  
        baseline (mu + b_u + b_i)               504         NaN  
        CF (user-based, k=10)                   504      2.5042

In [8]:
# Train / validation / test for the final CF models (checks for over-fitting)
cf_rows = results_df[results_df["model"].str.startswith("CF")]
cf_rows.pivot_table(index=["dataset", "model"], columns="split", values=["MSE", "CI"]).round(4)

CI                     MSE                
split                            test   train     val    test   train     val
dataset model                                                                
DAVIS   CF (item-based, k=10)  0.8443  0.9025  0.8565  0.2774  0.1706  0.3089
KIBA    CF (user-based, k=10)  0.8456  0.8797  0.8465  0.2202  0.1337  0.2237

### Where does CF help, and where does it fall back?

DAVIS: error on screening-limit pairs ($\mathrm{p}K_d = 5$, censored "weaker than measurable") vs. measured binders. 

KIBA: how many neighbours each test prediction could use, as well as whether ligands with few training observations get worse predictions.

In [9]:
d = scored_test["davis"]
at_limit = np.isclose(d["score"], SCALES["davis"]["screening_limit"])
base_davis = fit_cf(splits["davis"][splits["davis"]["split"] == "train"], *build_index(splits["davis"]), "baseline")
d_base, _ = predict(base_davis, d["ligand_key"], d["target_key"])
davis_breakdown = pd.DataFrame({
    "test pairs": [at_limit.sum(), (~at_limit).sum()],
    "CF MSE": [np.mean((d["score"] - d["pred"])[at_limit] ** 2), np.mean((d["score"] - d["pred"])[~at_limit] ** 2)],
    "baseline MSE": [np.mean((d["score"] - d_base)[at_limit] ** 2), np.mean((d["score"] - d_base)[~at_limit] ** 2)],
}, index=["at screening limit (pKd = 5)", "measured binding (pKd > 5)"]).round(4)
display(davis_breakdown)

k_df = scored_test["kiba"]
kiba_train = splits["kiba"][splits["kiba"]["split"] == "train"]
n_train_obs = kiba_train.groupby("ligand_key").size()
k_df = k_df.assign(ligand_train_obs=k_df["ligand_key"].map(n_train_obs))
k_df["ligand training observations"] = pd.cut(k_df["ligand_train_obs"], [0, 10, 20, 50, 100, 1000])
kiba_breakdown = k_df.groupby("ligand training observations", observed=True).agg(
    test_pairs=("score", "size"),
    CF_MSE=("pred", lambda s: np.mean((k_df.loc[s.index, "score"] - s) ** 2)),
    mean_neighbours=("n_neighbours", "mean"),
    fallback_pct=("n_neighbours", lambda s: 100 * np.mean(s == 0)),
).round(3)
kiba_breakdown

,test pairs,CF MSE,baseline MSE
at screening limit (pKd = 5),2790,0.0872,0.1677
measured binding (pKd > 5),1077,0.7702,1.0273


,test_pairs,CF_MSE,mean_neighbours,fallback_pct
ligand training observations,,,,
"(0, 10]",1202,0.506,7.372,25.707
"(10, 20]",1224,0.439,9.136,8.170
"(20, 50]",2976,0.237,9.870,1.109
"(50, 100]",10634,0.161,9.997,0.000
"(100, 1000]",1614,0.204,9.989,0.000


## Step 6: Save the models and verify reloading

Each checkpoint stores the full CF state (ligand/target vocabularies, $\mu$, biases, training deviation matrix, similarity matrix, hyperparameters) and the score-scale information. 

In [10]:
from project2_cf import save_model, load_model

for name, model in final_models.items():
    path = MODEL_DIR / f"{name}_cf.npz"
    save_model(model, path, SCALES[name], seed=SEED)
    reloaded = load_model(path)
    test = splits[name][splits[name]["split"] == "test"]
    before, _ = predict(model, test["ligand_key"], test["target_key"])
    after, _ = predict(reloaded, test["ligand_key"], test["target_key"])
    print(f"{path}: {path.stat().st_size / 1e6:.1f} MB, scale = {reloaded['scale']['score_name']}, "
          f"identical predictions after reload: {np.array_equal(before, after)}")

saved_models\davis_cf.npz: 1.2 MB, scale = pKd, identical predictions after reload: True


saved_models\kiba_cf.npz: 24.9 MB, scale = KIBA score, identical predictions after reload: True


## Step 7: Fixed case studies from reloaded models

We retain the six ligand IDs already used in the original notebook, which match the example IDs in the project brief. Keeping this list unchanged makes the comparison reproducible; no new cases were selected based on the extension results. Each selected ID has training observations and held-out test candidates in the seed-42 split. These six examples illustrate behaviour, but are not a representative random sample.

All predictions below come from models reloaded from disk. The best measured test target(s) are labelled **correct** and other candidates **wrong**, following the assignment convention; "wrong" does not imply that the target cannot bind. Display order uses the target ID to break equal predicted scores, while **evaluation integrates over all tie orders**. Minimum/maximum rank columns make ties visible.


In [11]:
CASES = {"davis": ["10127622", "11338033", "44259"],
         "kiba": ["CHEMBL1967538", "CHEMBL2004544", "CHEMBL565612"]}
loaded = {name: load_model(MODEL_DIR / f"{name}_cf.npz") for name in CASES}
case_summary = []
for name, ligs in CASES.items():
    df = splits[name]
    test = df[df["split"] == "test"]
    model = loaded[name]
    for lig in ligs:
        g = test[test["ligand_key"] == lig].copy()
        n_train = int(((df["split"] == "train") & (df["ligand_key"] == lig)).sum())
        if g.empty:
            print(f"{name.upper()} {lig}: no test candidates in this split; case excluded.")
            continue
        pred, n_nb = predict(model, g["ligand_key"], g["target_key"])
        g = g.assign(pred=pred, neighbours=n_nb)
        g["rank min"] = g["pred"].rank(method="min", ascending=False).astype(int)
        g["rank max"] = g["pred"].rank(method="max", ascending=False).astype(int)
        best = g["score"] == g["score"].max()
        g["label"] = np.where(best, "correct", "wrong")
        metrics = summarize_predictions(g)
        rk = ranking_metrics(g)
        expected_rank = rk.iloc[0]["rank_of_best"] if len(rk) else np.nan
        case_summary.append({
            "dataset": name.upper(), "ligand": lig, "train obs": n_train,
            "test targets": len(g), "true best score": g["score"].max(),
            "tied best targets": int(best.sum()), "MSE": metrics["MSE"], "CI": metrics["CI"],
            "expected rank of true best": expected_rank, "Hit@1": metrics["Hit@1"],
            "Hit@5": metrics["Hit@5"], "random Hit@5": metrics["random Hit@5"],
            "fallback %": 100 * np.mean(n_nb == 0),
        })
        print(f"\n{name.upper()} ligand {lig} ({n_train} training scores, {len(g)} test targets)")
        display(g.sort_values(["pred", "target_key"], ascending=[False, True])[
            ["rank min", "rank max", "target_key", "pred", "score", "label", "neighbours"]]
            .rename(columns={"target_key": "target", "pred": "predicted", "score": "true"})
            .head(10).round(3).reset_index(drop=True))
        print(f"Interpretation: expected first-best rank {expected_rank:.2f}; Hit@5 {metrics['Hit@5']:.3f} "
              f"versus random {metrics['random Hit@5']:.3f}; within-ligand CI {metrics['CI']:.3f}. "
              f"{int((n_nb == 0).sum())}/{len(g)} predictions used the baseline fallback. "
              "These rankings concern only measured held-out candidates, not every possible protein.")
pd.DataFrame(case_summary).round(3)



DAVIS ligand 10127622 (270 training scores, 65 test targets)


,rank min,rank max,target,predicted,true,label,neighbours
0,1,1,EPHB6,5.769,5.000,wrong,10
1,2,2,PFCDPK1(Pfalciparum),5.297,5.000,wrong,10
2,3,3,BMPR1B,5.247,5.000,wrong,10
3,4,4,MAP4K2,5.237,5.000,wrong,10
4,5,5,MEK1,5.188,7.004,correct,10
5,6,6,EPHA6,5.188,5.000,wrong,10
6,7,7,DRAK1,5.169,5.000,wrong,10
7,8,8,PRKCE,5.150,5.000,wrong,10
8,9,9,TTK,5.149,5.000,wrong,10
9,10,10,FGR,5.145,5.000,wrong,10


Interpretation: expected first-best rank 5.00; Hit@5 1.000 versus random 0.077; within-ligand CI 0.938. 0/65 predictions used the baseline fallback. These rankings concern only measured held-out candidates, not every possible protein.

DAVIS ligand 11338033 (268 training scores, 58 test targets)


,rank min,rank max,target,predicted,true,label,neighbours
0,1,1,CDKL5,6.903,8.585,correct,10
1,2,2,CDC2L2,6.692,8.284,wrong,10
2,3,3,PDGFRB,5.991,5.000,wrong,10
3,4,4,YSK4,5.779,5.000,wrong,10
4,5,5,PFPK5(Pfalciparum),5.740,5.000,wrong,10
5,6,6,HIPK3,5.719,5.000,wrong,10
6,7,7,JAK3(JH1domain-catalytic),5.531,5.000,wrong,10
7,8,8,CLK2,5.440,5.770,wrong,10
8,9,9,MAP4K4,5.412,5.000,wrong,10
9,10,10,CLK1,5.348,6.004,wrong,10


Interpretation: expected first-best rank 1.00; Hit@5 1.000 versus random 0.086; within-ligand CI 0.799. 0/58 predictions used the baseline fallback. These rankings concern only measured held-out candidates, not every possible protein.

DAVIS ligand 44259 (267 training scores, 56 test targets)


,rank min,rank max,target,predicted,true,label,neighbours
0,1,1,TAK1,8.224,7.174,wrong,10
1,2,2,JAK2(JH1domain-catalytic),8.221,8.009,wrong,10
2,3,3,PDGFRB,7.970,8.745,wrong,10
3,4,4,TRKA,7.890,8.398,wrong,10
4,5,5,DCAMKL3,7.848,7.770,wrong,10
5,6,6,CAMKK1,7.662,10.409,correct,10
6,7,7,SNARK,7.658,10.066,wrong,10
7,8,8,PRKX,7.623,7.886,wrong,10
8,9,9,TYK2(JH1domain-catalytic),7.607,6.380,wrong,10
9,10,10,PHKG1,7.565,9.187,wrong,10


... (4 outputs left out)

Interpretation: expected first-best rank 1.00; Hit@5 1.000 versus random 0.312; within-ligand CI 0.894. 0/16 predictions used the baseline fallback. These rankings concern only measured held-out candidates, not every possible protein.

KIBA ligand CHEMBL565612 (121 training scores, 27 test targets)


,rank min,rank max,target,predicted,true,label,neighbours
0,1,1,Q04759,13.552,15.269,correct,10
1,2,2,O00311,12.858,12.000,wrong,10
2,3,3,Q86V86,12.580,12.600,wrong,10
3,4,4,Q9Y463,12.389,11.700,wrong,10
4,5,5,P51817,12.374,11.700,wrong,10
5,6,6,O00329,12.374,11.100,wrong,10
6,7,7,P31751,12.019,11.600,wrong,10
7,8,8,P30530,11.984,11.141,wrong,10
8,9,9,P41743,11.935,12.500,wrong,10
9,10,10,P48736,11.913,11.100,wrong,10


Interpretation: expected first-best rank 1.00; Hit@5 1.000 versus random 0.185; within-ligand CI 0.630. 0/27 predictions used the baseline fallback. These rankings concern only measured held-out candidates, not every possible protein.


,dataset,ligand,train obs,test targets,true best score,tied best targets,MSE,CI,expected rank of true best,Hit@1,Hit@5,random Hit@5,fallback %
0,DAVIS,10127622,270,65,7.004,1,0.078,0.938,5.0,0.0,1.0,0.077,0.0
1,DAVIS,11338033,268,58,8.585,1,0.516,0.799,1.0,1.0,1.0,0.086,0.0
2,DAVIS,44259,267,56,10.409,1,0.990,0.775,6.0,0.0,0.0,0.089,0.0
3,KIBA,CHEMBL1967538,15,8,11.700,1,0.181,0.769,2.0,0.0,1.0,0.625,0.0
4,KIBA,CHEMBL2004544,71,16,12.500,1,0.084,0.894,1.0,1.0,1.0,0.312,0.0
5,KIBA,CHEMBL565612,121,27,15.269,1,0.362,0.630,1.0,1.0,1.0,0.185,0.0
